# 01 — Data loading and preprocessing

This notebook loads the raw TCGA-BRCA cBioPortal files, inspects the available clinical and molecular variables, harmonizes sample identifiers and saves clean processed tables for downstream analysis.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

# Allow imports from src/
PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from data_utils import (
    read_cbioportal_table,
    clean_expression_matrix,
    harmonize_tcga_sample_id,
    harmonize_tcga_patient_id,
    find_columns_containing,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

In [2]:
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "cbioportal_brca"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

clinical_patient_path = RAW_DIR / "data_clinical_patient.txt"
clinical_sample_path = RAW_DIR / "data_clinical_sample.txt"
expression_path = RAW_DIR / "data_mrna_seq_v2_rsem.txt"

for path in [clinical_patient_path, clinical_sample_path, expression_path]:
    print(path, "exists:", path.exists())

/Users/rubensiok/tcga-brca-translational-stratification/data/raw/cbioportal_brca/data_clinical_patient.txt exists: True
/Users/rubensiok/tcga-brca-translational-stratification/data/raw/cbioportal_brca/data_clinical_sample.txt exists: True
/Users/rubensiok/tcga-brca-translational-stratification/data/raw/cbioportal_brca/data_mrna_seq_v2_rsem.txt exists: True


In [3]:
clinical_patient = read_cbioportal_table(clinical_patient_path)
clinical_sample = read_cbioportal_table(clinical_sample_path)
expression_raw = read_cbioportal_table(expression_path)

print("clinical_patient:", clinical_patient.shape)
print("clinical_sample:", clinical_sample.shape)
print("expression_raw:", expression_raw.shape)

clinical_patient: (1084, 38)
clinical_sample: (1084, 19)
expression_raw: (20531, 1084)


In [4]:
print("Clinical patient columns:")
display(pd.DataFrame({"column": clinical_patient.columns}))

print("Clinical sample columns:")
display(pd.DataFrame({"column": clinical_sample.columns}))

Clinical patient columns:


,column
0,PATIENT_ID
1,SUBTYPE
2,CANCER_TYPE_ACRONYM
3,OTHER_PATIENT_ID
4,AGE
5,SEX
6,AJCC_PATHOLOGIC_TUMOR_STAGE
7,AJCC_STAGING_EDITION
8,DAYS_LAST_FOLLOWUP
9,DAYS_TO_BIRTH


Clinical sample columns:


,column
0,PATIENT_ID
1,SAMPLE_ID
2,ONCOTREE_CODE
3,CANCER_TYPE
4,CANCER_TYPE_DETAILED
5,TUMOR_TYPE
6,GRADE
7,TISSUE_PROSPECTIVE_COLLECTION_INDICATOR
8,TISSUE_RETROSPECTIVE_COLLECTION_INDICATOR
9,TISSUE_SOURCE_SITE_CODE


In [5]:
keywords = [
    "SUBTYPE",
    "PAM50",
    "ER",
    "PR",
    "HER2",
    "STATUS",
    "SURVIVAL",
    "OS",
    "DFS",
    "TUMOR",
    "STAGE",
    "GRADE",
    "AGE",
]

patient_matches = find_columns_containing(clinical_patient, keywords)
sample_matches = find_columns_containing(clinical_sample, keywords)

print("Potentially useful patient-level columns:")
display(pd.DataFrame({"column": patient_matches}))

print("Potentially useful sample-level columns:")
display(pd.DataFrame({"column": sample_matches}))

Potentially useful patient-level columns:


,column
0,SUBTYPE
1,CANCER_TYPE_ACRONYM
2,OTHER_PATIENT_ID
3,AGE
4,AJCC_PATHOLOGIC_TUMOR_STAGE
5,DAYS_TO_INITIAL_PATHOLOGIC_DIAGNOSIS
6,INFORMED_CONSENT_VERIFIED
7,NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT
8,PATH_M_STAGE
9,PATH_N_STAGE


Potentially useful sample-level columns:


,column
0,CANCER_TYPE
1,CANCER_TYPE_DETAILED
2,TUMOR_TYPE
3,GRADE
4,TISSUE_PROSPECTIVE_COLLECTION_INDICATOR
5,TISSUE_RETROSPECTIVE_COLLECTION_INDICATOR
6,TUMOR_TISSUE_SITE
7,SOMATIC_STATUS


In [6]:
expression = clean_expression_matrix(expression_raw)

print("Clean expression matrix:")
print(expression.shape)
display(expression.iloc[:5, :5])

Clean expression matrix:
(1082, 20511)


Hugo_Symbol,A1BG,A1CF,A2BP1,A2LD1,A2M
SAMPLE_ID,,,,,
TCGA-3C-AAAU-01,197.090,0.0000,0.0000,102.9630,5798.37
TCGA-3C-AALI-01,237.384,0.0000,0.0000,70.8646,7571.98
TCGA-3C-AALJ-01,423.237,0.9066,0.0000,161.2600,8840.40
TCGA-3C-AALK-01,191.018,0.0000,0.0000,62.5072,10960.20
TCGA-4H-AAAK-01,268.881,0.4255,3.8298,154.3700,9585.44


In [7]:
clinical_sample = clinical_sample.copy()
clinical_patient = clinical_patient.copy()
expression = expression.copy()

# Standard cBioPortal columns are often SAMPLE_ID and PATIENT_ID
print("clinical_sample ID columns:")
display(clinical_sample[[col for col in clinical_sample.columns if "ID" in col.upper()]].head())

print("clinical_patient ID columns:")
display(clinical_patient[[col for col in clinical_patient.columns if "ID" in col.upper()]].head())

clinical_sample ID columns:


,PATIENT_ID,SAMPLE_ID,ANEUPLOIDY_SCORE
0,TCGA-3C-AAAU,TCGA-3C-AAAU-01,19.0
1,TCGA-3C-AALI,TCGA-3C-AALI-01,22.0
2,TCGA-3C-AALJ,TCGA-3C-AALJ-01,13.0
3,TCGA-3C-AALK,TCGA-3C-AALK-01,4.0
4,TCGA-4H-AAAK,TCGA-4H-AAAK-01,7.0


clinical_patient ID columns:


,PATIENT_ID,OTHER_PATIENT_ID
0,TCGA-3C-AAAU,6E7D5EC6-A469-467C-B748-237353C23416
1,TCGA-3C-AALI,55262FCB-1B01-4480-B322-36570430C917
2,TCGA-3C-AALJ,427D0648-3F77-4FFC-B52C-89855426D647
3,TCGA-3C-AALK,C31900A4-5DCD-4022-97AC-638E86E889E4
4,TCGA-4H-AAAK,6623FC5E-00BE-4476-967A-CBD55F676EA6


In [8]:
expression = expression.reset_index()

expression["SAMPLE_ID_ORIGINAL"] = expression["SAMPLE_ID"]
expression["SAMPLE_ID_15"] = expression["SAMPLE_ID_ORIGINAL"].apply(lambda x: harmonize_tcga_sample_id(x, n=15))
expression["PATIENT_ID_12"] = expression["SAMPLE_ID_ORIGINAL"].apply(harmonize_tcga_patient_id)

display(expression[["SAMPLE_ID_ORIGINAL", "SAMPLE_ID_15", "PATIENT_ID_12"]].head())

Hugo_Symbol,SAMPLE_ID_ORIGINAL,SAMPLE_ID_15,PATIENT_ID_12
0,TCGA-3C-AAAU-01,TCGA-3C-AAAU-01,TCGA-3C-AAAU
1,TCGA-3C-AALI-01,TCGA-3C-AALI-01,TCGA-3C-AALI
2,TCGA-3C-AALJ-01,TCGA-3C-AALJ-01,TCGA-3C-AALJ
3,TCGA-3C-AALK-01,TCGA-3C-AALK-01,TCGA-3C-AALK
4,TCGA-4H-AAAK-01,TCGA-4H-AAAK-01,TCGA-4H-AAAK


In [9]:
clinical_sample["SAMPLE_ID_ORIGINAL"] = clinical_sample["SAMPLE_ID"]
clinical_sample["SAMPLE_ID_15"] = clinical_sample["SAMPLE_ID"].apply(lambda x: harmonize_tcga_sample_id(x, n=15))

if "PATIENT_ID" in clinical_sample.columns:
    clinical_sample["PATIENT_ID_12"] = clinical_sample["PATIENT_ID"].apply(harmonize_tcga_patient_id)
else:
    clinical_sample["PATIENT_ID_12"] = clinical_sample["SAMPLE_ID"].apply(harmonize_tcga_patient_id)

display(clinical_sample[["SAMPLE_ID_ORIGINAL", "SAMPLE_ID_15", "PATIENT_ID_12"]].head())

,SAMPLE_ID_ORIGINAL,SAMPLE_ID_15,PATIENT_ID_12
0,TCGA-3C-AAAU-01,TCGA-3C-AAAU-01,TCGA-3C-AAAU
1,TCGA-3C-AALI-01,TCGA-3C-AALI-01,TCGA-3C-AALI
2,TCGA-3C-AALJ-01,TCGA-3C-AALJ-01,TCGA-3C-AALJ
3,TCGA-3C-AALK-01,TCGA-3C-AALK-01,TCGA-3C-AALK
4,TCGA-4H-AAAK-01,TCGA-4H-AAAK-01,TCGA-4H-AAAK


In [10]:
if "PATIENT_ID" not in clinical_patient.columns:
    raise ValueError("Expected PATIENT_ID column not found in clinical_patient.")

clinical_patient["PATIENT_ID_12"] = clinical_patient["PATIENT_ID"].apply(harmonize_tcga_patient_id)

display(clinical_patient[["PATIENT_ID", "PATIENT_ID_12"]].head())

,PATIENT_ID,PATIENT_ID_12
0,TCGA-3C-AAAU,TCGA-3C-AAAU
1,TCGA-3C-AALI,TCGA-3C-AALI
2,TCGA-3C-AALJ,TCGA-3C-AALJ
3,TCGA-3C-AALK,TCGA-3C-AALK
4,TCGA-4H-AAAK,TCGA-4H-AAAK


In [11]:
expression_samples = set(expression["SAMPLE_ID_15"])
clinical_samples = set(clinical_sample["SAMPLE_ID_15"])

sample_overlap = expression_samples.intersection(clinical_samples)

print("Expression samples:", len(expression_samples))
print("Clinical samples:", len(clinical_samples))
print("Overlap:", len(sample_overlap))
print("Expression only:", len(expression_samples - clinical_samples))
print("Clinical only:", len(clinical_samples - expression_samples))

Expression samples: 1082
Clinical samples: 1084
Overlap: 1082
Expression only: 0
Clinical only: 2


In [12]:
# Merge sample-level clinical metadata
clinical_merged = clinical_sample.merge(
    clinical_patient,
    on="PATIENT_ID_12",
    how="left",
    suffixes=("_sample", "_patient")
)

print("clinical_merged:", clinical_merged.shape)
display(clinical_merged.head())

clinical_merged: (1084, 60)


,PATIENT_ID_sample,SAMPLE_ID,ONCOTREE_CODE,CANCER_TYPE,CANCER_TYPE_DETAILED,TUMOR_TYPE,GRADE,TISSUE_PROSPECTIVE_COLLECTION_INDICATOR,TISSUE_RETROSPECTIVE_COLLECTION_INDICATOR,TISSUE_SOURCE_SITE_CODE,TUMOR_TISSUE_SITE,ANEUPLOIDY_SCORE,SAMPLE_TYPE,MSI_SCORE_MANTIS,MSI_SENSOR_SCORE,SOMATIC_STATUS,TMB_NONSYNONYMOUS,TISSUE_SOURCE_SITE,TBL_SCORE,SAMPLE_ID_ORIGINAL,SAMPLE_ID_15,PATIENT_ID_12,PATIENT_ID_patient,SUBTYPE,CANCER_TYPE_ACRONYM,OTHER_PATIENT_ID,AGE,SEX,AJCC_PATHOLOGIC_TUMOR_STAGE,AJCC_STAGING_EDITION,DAYS_LAST_FOLLOWUP,DAYS_TO_BIRTH,DAYS_TO_INITIAL_PATHOLOGIC_DIAGNOSIS,ETHNICITY,FORM_COMPLETION_DATE,HISTORY_NEOADJUVANT_TRTYN,ICD_10,ICD_O_3_HISTOLOGY,ICD_O_3_SITE,INFORMED_CONSENT_VERIFIED,NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT,PATH_M_STAGE,PATH_N_STAGE,PATH_T_STAGE,PERSON_NEOPLASM_CANCER_STATUS,PRIMARY_LYMPH_NODE_PRESENTATION_ASSESSMENT,PRIOR_DX,RACE,RADIATION_THERAPY,WEIGHT,IN_PANCANPATHWAYS_FREEZE,OS_STATUS,OS_MONTHS,DSS_STATUS,DSS_MONTHS,DFS_STATUS,DFS_MONTHS,PFS_STATUS,PFS_MONTHS,GENETIC_ANCESTRY_LABEL
0,TCGA-3C-AAAU,TCGA-3C-AAAU-01,ILC,Breast Cancer,Breast Invasive Lobular Carcinoma,Infiltrating Lobular Carcinoma,NaN,No,Yes,3C,Breast,19.0,Primary,0.3319,0.55,Matched,0.800000,Columbia University,205.0,TCGA-3C-AAAU-01,TCGA-3C-AAAU-01,TCGA-3C-AAAU,TCGA-3C-AAAU,BRCA_LumA,BRCA,6E7D5EC6-A469-467C-B748-237353C23416,55,Female,STAGE X,6TH,4047.0,-20211.0,0,Not Hispanic Or Latino,1/13/14,No,C50.9,8520/3,C50.9,Yes,No,MX,NX,TX,With Tumor,Yes,No,White,No,NaN,Yes,0:LIVING,133.050597,0:ALIVE OR DEAD TUMOR FREE,133.050597,1:Recurred/Progressed,59.440444,1:PROGRESSION,59.440444,EUR
1,TCGA-3C-AALI,TCGA-3C-AALI-01,IDC,Breast Cancer,Breast Invasive Ductal Carcinoma,Infiltrating Ductal Carcinoma,NaN,No,Yes,3C,Breast,22.0,Primary,0.3449,0.74,Matched,15.266667,Columbia University,190.0,TCGA-3C-AALI-01,TCGA-3C-AALI-01,TCGA-3C-AALI,TCGA-3C-AALI,BRCA_Her2,BRCA,55262FCB-1B01-4480-B322-36570430C917,50,Female,STAGE IIB,6TH,4005.0,-18538.0,0,Not Hispanic Or Latino,7/28/14,No,C50.9,8500/3,C50.9,Yes,No,M0,N1A,T2,Tumor Free,Yes,No,Black or African American,Yes,NaN,Yes,0:LIVING,131.669790,0:ALIVE OR DEAD TUMOR FREE,131.669790,0:DiseaseFree,131.669790,0:CENSORED,131.669790,AFR
2,TCGA-3C-AALJ,TCGA-3C-AALJ-01,IDC,Breast Cancer,Breast Invasive Ductal Carcinoma,Infiltrating Ductal Carcinoma,NaN,No,Yes,3C,Breast,13.0,Primary,0.3266,0.31,Matched,0.933333,Columbia University,365.0,TCGA-3C-AALJ-01,TCGA-3C-AALJ-01,TCGA-3C-AALJ,TCGA-3C-AALJ,BRCA_LumB,BRCA,427D0648-3F77-4FFC-B52C-89855426D647,62,Female,STAGE IIB,7TH,1474.0,-22848.0,0,Not Hispanic Or Latino,7/28/14,No,C50.9,8500/3,C50.9,Yes,No,M0,N1A,T2,Tumor Free,Yes,No,Black or African American,No,NaN,Yes,0:LIVING,48.459743,0:ALIVE OR DEAD TUMOR FREE,48.459743,0:DiseaseFree,48.459743,0:CENSORED,48.459743,AFR_ADMIX
3,TCGA-3C-AALK,TCGA-3C-AALK-01,IDC,Breast Cancer,Breast Invasive Ductal Carcinoma,Infiltrating Ductal Carcinoma,NaN,No,Yes,3C,Breast,4.0,Primary,0.3218,0.03,Matched,1.500000,Columbia University,25.0,TCGA-3C-AALK-01,TCGA-3C-AALK-01,TCGA-3C-AALK,TCGA-3C-AALK,BRCA_LumA,BRCA,C31900A4-5DCD-4022-97AC-638E86E889E4,52,Female,STAGE IA,7TH,1448.0,-19074.0,0,Not Hispanic Or Latino,7/28/14,No,C50.9,8500/3,C50.9,Yes,No,M0,N0 (I+),T1C,Tumor Free,Yes,No,Black or African American,No,NaN,Yes,0:LIVING,47.604958,0:ALIVE OR DEAD TUMOR FREE,47.604958,NaN,NaN,0:CENSORED,47.604958,AFR
4,TCGA-4H-AAAK,TCGA-4H-AAAK-01,ILC,Breast Cancer,Breast Invasive Lobular Carcinoma,Infiltrating Lobular Carcinoma,NaN,Yes,No,4H,Breast,7.0,Primary,0.3411,0.01,Matched,0.700000,"Proteogenex, Inc.",36.0,TCGA-4H-AAAK-01,TCGA-4H-AAAK-01,TCGA-4H-AAAK,TCGA-4H-AAAK,BRCA_LumA,BRCA,6623FC5E-00BE-4476-967A-CBD55F676EA6,50,Female,STAGE IIIA,7TH,348.0,-18371.0,0,Not Hispanic Or Latino,11/13/14,No,C50.9,8520/3,C50.9,Yes,No,M0,N2A,T2,Tumor Free,Yes,No,White,No,NaN,Yes,0:LIVING,11.440971,0:ALIVE OR DEAD TUMOR FREE,11.440971,0:DiseaseFree,11.440971,0:CENSORED,11.440971,EUR


In [13]:
metadata_expression = clinical_merged.merge(
    expression[["SAMPLE_ID_ORIGINAL", "SAMPLE_ID_15", "PATIENT_ID_12"]],
    on=["SAMPLE_ID_15", "PATIENT_ID_12"],
    how="inner"
)

print("metadata_expression:", metadata_expression.shape)
display(metadata_expression.head())

metadata_expression: (1082, 61)


,PATIENT_ID_sample,SAMPLE_ID,ONCOTREE_CODE,CANCER_TYPE,CANCER_TYPE_DETAILED,TUMOR_TYPE,GRADE,TISSUE_PROSPECTIVE_COLLECTION_INDICATOR,TISSUE_RETROSPECTIVE_COLLECTION_INDICATOR,TISSUE_SOURCE_SITE_CODE,TUMOR_TISSUE_SITE,ANEUPLOIDY_SCORE,SAMPLE_TYPE,MSI_SCORE_MANTIS,MSI_SENSOR_SCORE,SOMATIC_STATUS,TMB_NONSYNONYMOUS,TISSUE_SOURCE_SITE,TBL_SCORE,SAMPLE_ID_ORIGINAL_x,SAMPLE_ID_15,PATIENT_ID_12,PATIENT_ID_patient,SUBTYPE,CANCER_TYPE_ACRONYM,OTHER_PATIENT_ID,AGE,SEX,AJCC_PATHOLOGIC_TUMOR_STAGE,AJCC_STAGING_EDITION,DAYS_LAST_FOLLOWUP,DAYS_TO_BIRTH,DAYS_TO_INITIAL_PATHOLOGIC_DIAGNOSIS,ETHNICITY,FORM_COMPLETION_DATE,HISTORY_NEOADJUVANT_TRTYN,ICD_10,ICD_O_3_HISTOLOGY,ICD_O_3_SITE,INFORMED_CONSENT_VERIFIED,NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT,PATH_M_STAGE,PATH_N_STAGE,PATH_T_STAGE,PERSON_NEOPLASM_CANCER_STATUS,PRIMARY_LYMPH_NODE_PRESENTATION_ASSESSMENT,PRIOR_DX,RACE,RADIATION_THERAPY,WEIGHT,IN_PANCANPATHWAYS_FREEZE,OS_STATUS,OS_MONTHS,DSS_STATUS,DSS_MONTHS,DFS_STATUS,DFS_MONTHS,PFS_STATUS,PFS_MONTHS,GENETIC_ANCESTRY_LABEL,SAMPLE_ID_ORIGINAL_y
0,TCGA-3C-AAAU,TCGA-3C-AAAU-01,ILC,Breast Cancer,Breast Invasive Lobular Carcinoma,Infiltrating Lobular Carcinoma,NaN,No,Yes,3C,Breast,19.0,Primary,0.3319,0.55,Matched,0.800000,Columbia University,205.0,TCGA-3C-AAAU-01,TCGA-3C-AAAU-01,TCGA-3C-AAAU,TCGA-3C-AAAU,BRCA_LumA,BRCA,6E7D5EC6-A469-467C-B748-237353C23416,55,Female,STAGE X,6TH,4047.0,-20211.0,0,Not Hispanic Or Latino,1/13/14,No,C50.9,8520/3,C50.9,Yes,No,MX,NX,TX,With Tumor,Yes,No,White,No,NaN,Yes,0:LIVING,133.050597,0:ALIVE OR DEAD TUMOR FREE,133.050597,1:Recurred/Progressed,59.440444,1:PROGRESSION,59.440444,EUR,TCGA-3C-AAAU-01
1,TCGA-3C-AALI,TCGA-3C-AALI-01,IDC,Breast Cancer,Breast Invasive Ductal Carcinoma,Infiltrating Ductal Carcinoma,NaN,No,Yes,3C,Breast,22.0,Primary,0.3449,0.74,Matched,15.266667,Columbia University,190.0,TCGA-3C-AALI-01,TCGA-3C-AALI-01,TCGA-3C-AALI,TCGA-3C-AALI,BRCA_Her2,BRCA,55262FCB-1B01-4480-B322-36570430C917,50,Female,STAGE IIB,6TH,4005.0,-18538.0,0,Not Hispanic Or Latino,7/28/14,No,C50.9,8500/3,C50.9,Yes,No,M0,N1A,T2,Tumor Free,Yes,No,Black or African American,Yes,NaN,Yes,0:LIVING,131.669790,0:ALIVE OR DEAD TUMOR FREE,131.669790,0:DiseaseFree,131.669790,0:CENSORED,131.669790,AFR,TCGA-3C-AALI-01
2,TCGA-3C-AALJ,TCGA-3C-AALJ-01,IDC,Breast Cancer,Breast Invasive Ductal Carcinoma,Infiltrating Ductal Carcinoma,NaN,No,Yes,3C,Breast,13.0,Primary,0.3266,0.31,Matched,0.933333,Columbia University,365.0,TCGA-3C-AALJ-01,TCGA-3C-AALJ-01,TCGA-3C-AALJ,TCGA-3C-AALJ,BRCA_LumB,BRCA,427D0648-3F77-4FFC-B52C-89855426D647,62,Female,STAGE IIB,7TH,1474.0,-22848.0,0,Not Hispanic Or Latino,7/28/14,No,C50.9,8500/3,C50.9,Yes,No,M0,N1A,T2,Tumor Free,Yes,No,Black or African American,No,NaN,Yes,0:LIVING,48.459743,0:ALIVE OR DEAD TUMOR FREE,48.459743,0:DiseaseFree,48.459743,0:CENSORED,48.459743,AFR_ADMIX,TCGA-3C-AALJ-01
3,TCGA-3C-AALK,TCGA-3C-AALK-01,IDC,Breast Cancer,Breast Invasive Ductal Carcinoma,Infiltrating Ductal Carcinoma,NaN,No,Yes,3C,Breast,4.0,Primary,0.3218,0.03,Matched,1.500000,Columbia University,25.0,TCGA-3C-AALK-01,TCGA-3C-AALK-01,TCGA-3C-AALK,TCGA-3C-AALK,BRCA_LumA,BRCA,C31900A4-5DCD-4022-97AC-638E86E889E4,52,Female,STAGE IA,7TH,1448.0,-19074.0,0,Not Hispanic Or Latino,7/28/14,No,C50.9,8500/3,C50.9,Yes,No,M0,N0 (I+),T1C,Tumor Free,Yes,No,Black or African American,No,NaN,Yes,0:LIVING,47.604958,0:ALIVE OR DEAD TUMOR FREE,47.604958,NaN,NaN,0:CENSORED,47.604958,AFR,TCGA-3C-AALK-01
4,TCGA-4H-AAAK,TCGA-4H-AAAK-01,ILC,Breast Cancer,Breast Invasive Lobular Carcinoma,Infiltrating Lobular Carcinoma,NaN,Yes,No,4H,Breast,7.0,Primary,0.3411,0.01,Matched,0.700000,"Proteogenex, Inc.",36.0,TCGA-4H-AAAK-01,TCGA-4H-AAAK-01,TCGA-4H-AAAK,TCGA-4H-AAAK,BRCA_LumA,BRCA,6623FC5E-00BE-4476-967A-CBD55F676EA6,50,Female,STAGE IIIA,7TH,348.0,-18371.0,0,Not Hispanic Or Latino,11/13/14,No,C50.9,8520/3,C50.9,Yes,No,M0,N2A,T2,Tumor Free,Yes,No,White,No,NaN,Yes,0:LIVING,11.440971,0:ALIVE OR DEAD TUMOR FREE,11.440971,0:DiseaseFree,11.440971,0:CENSORED,11.440971,EUR,TC

In [14]:
gene_columns = [
    col for col in expression.columns
    if col not in ["SAMPLE_ID", "SAMPLE_ID_ORIGINAL", "SAMPLE_ID_15", "PATIENT_ID_12"]
]

expression_clean = expression[["SAMPLE_ID_ORIGINAL", "SAMPLE_ID_15", "PATIENT_ID_12"] + gene_columns].copy()

print("expression_clean:", expression_clean.shape)
display(expression_clean.iloc[:5, :8])

expression_clean: (1082, 20514)


Hugo_Symbol,SAMPLE_ID_ORIGINAL,SAMPLE_ID_15,PATIENT_ID_12,A1BG,A1CF,A2BP1,A2LD1,A2M
0,TCGA-3C-AAAU-01,TCGA-3C-AAAU-01,TCGA-3C-AAAU,197.090,0.0000,0.0000,102.9630,5798.37
1,TCGA-3C-AALI-01,TCGA-3C-AALI-01,TCGA-3C-AALI,237.384,0.0000,0.0000,70.8646,7571.98
2,TCGA-3C-AALJ-01,TCGA-3C-AALJ-01,TCGA-3C-AALJ,423.237,0.9066,0.0000,161.2600,8840.40
3,TCGA-3C-AALK-01,TCGA-3C-AALK-01,TCGA-3C-AALK,191.018,0.0000,0.0000,62.5072,10960.20
4,TCGA-4H-AAAK-01,TCGA-4H-AAAK-01,TCGA-4H-AAAK,268.881,0.4255,3.8298,154.3700,9585.44


In [15]:
marker_genes = [
    "ESR1",
    "PGR",
    "ERBB2",
    "MKI67",
    "FOXA1",
    "GATA3",
    "TP53",
    "PIK3CA",
    "BRCA1",
    "BRCA2",
]

available_markers = [gene for gene in marker_genes if gene in expression_clean.columns]
missing_markers = [gene for gene in marker_genes if gene not in expression_clean.columns]

print("Available markers:", available_markers)
print("Missing markers:", missing_markers)

marker_expression = expression_clean[
    ["SAMPLE_ID_ORIGINAL", "SAMPLE_ID_15", "PATIENT_ID_12"] + available_markers
].copy()

display(marker_expression.head())

Available markers: ['ESR1', 'PGR', 'ERBB2', 'MKI67', 'FOXA1', 'GATA3', 'TP53', 'PIK3CA', 'BRCA1', 'BRCA2']
Missing markers: []


Hugo_Symbol,SAMPLE_ID_ORIGINAL,SAMPLE_ID_15,PATIENT_ID_12,ESR1,PGR,ERBB2,MKI67,FOXA1,GATA3,TP53,PIK3CA,BRCA1,BRCA2
0,TCGA-3C-AAAU-01,TCGA-3C-AAAU-01,TCGA-3C-AAAU,3457.9600,2273.2700,7113.41,2582.870,5448.37,14337.50,1385.530,487.003,831.317,178.878
1,TCGA-3C-AALI-01,TCGA-3C-AALI-01,TCGA-3C-AALI,68.5155,27.1887,194625.00,2285.480,6049.48,7437.74,414.356,321.914,389.886,153.888
2,TCGA-3C-AALJ-01,TCGA-3C-AALJ-01,TCGA-3C-AALJ,7482.3200,473.8350,11070.70,949.229,4620.13,10252.90,1289.210,216.682,200.363,151.405
3,TCGA-3C-AALK-01,TCGA-3C-AALK-01,TCGA-3C-AALK,2485.3100,2236.4600,36022.80,1139.430,7352.09,8761.69,1418.290,326.024,148.118,102.193
4,TCGA-4H-AAAK-01,TCGA-4H-AAAK-01,TCGA-4H-AAAK,5518.3000,4425.1900,12236.60,1206.380,6799.57,14068.50,1484.260,410.213,375.745,128.936


In [16]:
clinical_patient.to_csv(PROCESSED_DIR / "clinical_patient.tsv", sep="\t", index=False)
clinical_sample.to_csv(PROCESSED_DIR / "clinical_sample.tsv", sep="\t", index=False)
clinical_merged.to_csv(PROCESSED_DIR / "clinical_merged.tsv", sep="\t", index=False)
metadata_expression.to_csv(PROCESSED_DIR / "metadata_expression_samples.tsv", sep="\t", index=False)
expression_clean.to_csv(PROCESSED_DIR / "expression_rsem_samples_by_gene.tsv", sep="\t", index=False)
marker_expression.to_csv(PROCESSED_DIR / "marker_expression.tsv", sep="\t", index=False)

print("Saved processed files to:", PROCESSED_DIR)

Saved processed files to: /Users/rubensiok/tcga-brca-translational-stratification/data/processed
